# Train 3B voice LoRA on Colab

Fresh LoRA on `Qwen/Qwen2.5-3B-Instruct` using the ETHICS **voice** CoTs.
Do **not** continue from `qwen3b-cot-sft-v2`.

**Before starting:** Runtime → Change runtime type → **T4 GPU** (A100 if you have it).

On your Mac, upload both:

- `data/training_data/synthetic_ethics_voice_cot_train.jsonl` (452)
- `data/validation_data/synthetic_ethics_voice_cot_val.jsonl` (48)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU, then rerun."
print(torch.cuda.get_device_name(0))

In [ ]:
REPO_URL = "https://github.com/vladflorinfilip/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography.git"

!rm -rf Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!git clone --depth 1 "{REPO_URL}"
%cd Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography

In [ ]:
!pip install -q peft "transformers<4.50" "datasets<4" accelerate pyyaml tqdm

In [ ]:
from google.colab import files
from pathlib import Path

paths = {
    "synthetic_ethics_voice_cot_train.jsonl": Path("data/training_data/synthetic_ethics_voice_cot_train.jsonl"),
    "synthetic_ethics_voice_cot_val.jsonl": Path("data/validation_data/synthetic_ethics_voice_cot_val.jsonl"),
}
for dest in paths.values():
    dest.parent.mkdir(parents=True, exist_ok=True)

print("Upload the train JSONL, then the val JSONL (select both).")
uploaded = files.upload()
for name, data in uploaded.items():
    dest = paths.get(Path(name).name)
    if dest is None:
        raise SystemExit(f"Unexpected file {name}. Upload the train and val JSONLs.")
    dest.write_bytes(data)
    n = sum(1 for line in dest.read_text(encoding="utf-8").splitlines() if line.strip())
    print(f"wrote {dest} n={n}")

missing = [p.name for p in paths.values() if not p.exists()]
if missing:
    raise SystemExit(f"Still missing: {missing}")

In [ ]:
# Optional: set a Colab secret named HF_TOKEN if the Qwen download asks for auth.
# from google.colab import userdata
# import os
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!python training/train.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --data data/training_data/synthetic_ethics_voice_cot_train.jsonl \
  --val-data data/validation_data/synthetic_ethics_voice_cot_val.jsonl \
  --output-dir checkpoints/qwen3b-cot-sft-voice \
  --lora \
  --val-fraction 0

In [ ]:
from google.colab import files
from pathlib import Path

out = Path("checkpoints/qwen3b-cot-sft-voice")
assert (out / "adapter_model.safetensors").exists(), out
zip_path = Path("/content/qwen3b-cot-sft-voice-minimal.zip")
!zip -j {zip_path} {out}/adapter_model.safetensors {out}/adapter_config.json {out}/training_log.json
files.download(str(zip_path))